# Logistic Regression Baseline

This notebook trains a first logistic regression baseline using the preprocessed and selected feature matrices from the earlier notebooks.


In [ ]:
# 1) setup and feature selection inputs

from pathlib import Path

project_root = Path.cwd()

if not (project_root / "notebooks" / "04_feature_selection.ipynb").exists():
    project_root = project_root.parent

feature_selection_notebook = project_root / "notebooks" / "04_feature_selection.ipynb"

%run "$feature_selection_notebook"


In [ ]:
# 2) imports and validation split

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_model = X_train_selected.copy()
y_model = y_train.copy()

X_train_part, X_valid_part, y_train_part, y_valid_part = train_test_split(
    X_model,
    y_model,
    test_size=0.20,
    random_state=42,
    stratify=y_model
)

print("X_train_part shape:", X_train_part.shape)
print("X_valid_part shape:", X_valid_part.shape)


In [ ]:
# 3) train logistic regression baseline

logistic_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(
            max_iter=1000,
            solver="lbfgs",
            random_state=42
        ))
    ]
)

logistic_model.fit(X_train_part, y_train_part)

valid_predictions = logistic_model.predict(X_valid_part)
valid_accuracy = accuracy_score(y_valid_part, valid_predictions)

print("validation accuracy:", round(valid_accuracy, 4))


In [ ]:
# 4) validation diagnostics

confusion = pd.DataFrame(
    confusion_matrix(y_valid_part, valid_predictions),
    index=["actual_0", "actual_1"],
    columns=["predicted_0", "predicted_1"]
)

display(confusion)

print(classification_report(y_valid_part, valid_predictions, digits=4))


In [ ]:
# 5) coefficient inspection

coefs = logistic_model.named_steps["logistic_regression"].coef_[0]

coef_importance = pd.DataFrame({
    "feature": X_model.columns,
    "coefficient": coefs,
    "absolute_coefficient": np.abs(coefs)
}).sort_values("absolute_coefficient", ascending=False)

display(coef_importance.head(30))


In [ ]:
# 6) train on all training rows and prepare test predictions

logistic_model.fit(X_model, y_model)

test_predictions = logistic_model.predict(X_test_selected).astype(int)

submission = pd.read_csv(project_root / "data" / "submission.csv")
logistic_submission = submission.copy()

prediction_col = "prediction"

if prediction_col not in logistic_submission.columns:
    prediction_col = logistic_submission.columns[-1]

logistic_submission[prediction_col] = test_predictions

print("test predictions:")
print(pd.Series(test_predictions).value_counts().sort_index())

display(logistic_submission.head())
